<h3><center><font size=10>Plant Seedling Classification</center></font></h3>

# **Problem Statement**

### **Background**

In recent times, the field of agriculture has been in urgent need of modernizing, since the amount of manual work people need to put in to check if plants are growing correctly is still highly extensive. Despite several advances in agricultural technology, people working in the agricultural industry still need to have the ability to sort and recognize different plants and weeds, which takes a lot of time and effort in the long term.

The potential is ripe for this trillion-dollar industry to be greatly impacted by technological innovations that cut down on the requirement for manual labor, and this is where Artificial Intelligence can benefit the workers in this field, as the time and energy required to identify plant seedlings will be greatly shortened by the use of AI and Deep Learning. The ability to do so far more efficiently and even more effectively than experienced manual labor could lead to better crop yields, the freeing up of human involvement for higher-order agricultural decision making, and in the long term will result in more sustainable environmental practices in agriculture as well.

### **Objective**

The Aarhus University Signal Processing group, in collaboration with the University of Southern Denmark, has provided the data containing images of unique plants belonging to 12 different species. You being a data scientist, need to build a Convolutional Neural Network model which would classify the plant seedlings into their respective 12 categories.

### **Data Description**

This dataset contains images of unique plants belonging to 12 different species.

The data file names are:
- images.npy
- Labels.csv
  
- Due to the large volume of data, the images were converted to numpy arrays and stored in images.npy file and the corresponding labels are also put into Labels.csv so that you can work on the data/project seamlessly without having to worry about the high data volume.
The goal of the project is to create a classifier capable of determining a plant's species from an image.
Sugar beet.

List of Plant species
- Black-grass
- Charlock
- Cleavers
- Common Chickweed
- Common Wheat
- Fat Hen
- Loose Silky-bent
- Maize
- Scentless Mayweed
- Shepherds Purse
- Small-flowered Cranesbill
- Sugar beet

### **Importing necessary libraries**

In [ ]:
#importing necessary libraries
from PIL import Image

#libraries for reading and data manipulating
import pandas as pd
import numpy as np

#libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

#libraries for data pre-processing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#libraries for model building
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator


#Other libraries
import random

### **Reading the dataset**

In [ ]:
data= np.load("images_plant.npy")
print(data.shape)

In [ ]:
data_labels= pd.read_csv("Labels_plant.csv")
data_labels.sample(5)

In [ ]:
data_labels.shape

## **Overview of the dataset**

### **Checking the shape of the dataset**

In [ ]:
flattened= data.reshape(4750,-1)
df= pd.DataFrame(flattened)
df.shape

In [ ]:
data_labels.shape

### **Summary of data**

In [ ]:
df.info()

In [ ]:
data_labels.info()

### **Viewing first five images**

In [ ]:
n= 5
fig, axes= plt.subplots(1, n, figsize= (12,5))
for i in range(n):
    axes[i].imshow(data[i])
    axes[i].axis("off")
    axes[i].set_title(f"Image {i}")
plt.show()

## **Exploratory Data Analysis**

In [ ]:
#finding unique categories
classes= np.unique(data_labels)
classes

In [ ]:
#Plot random images from each of the classes and print their corresponding labels
fig, axes= plt.subplots(4, 3, figsize=(12,10))
axes= axes.flatten()

for i, cls in enumerate(classes):
    idx= np.where(data_labels == cls)[0]
    rand_idx= random.choice(idx)

    axes[i].imshow(data[rand_idx])
    axes[i].axis("off")
    axes[i].set_title(f"Class{cls}")
plt.show()

In [ ]:
#count plot for each category
classes1, counts= np.unique(data_labels, return_counts= True)
ax= sns.barplot(x= classes1, y= counts)
ax.bar_label(ax.containers[0])
plt.xticks(rotation= 90)
plt.show()

### **Data Preprocessing**

In [ ]:
#BGR image
plt.imshow(data[0])
plt.axis("off")
plt.show()

In [ ]:
#Convert the BGR images to RGB images
data_rgb= data[...,: : -1]
plt.imshow(data_rgb[0])
plt.axis("off")
plt.show()

In [ ]:
n= 6

fig, axes= plt.subplots(2, n, figsize= (15,5))
for i in range(n):

  #Before pre-processing
  axes[0,i].imshow(data[i])
  axes[0,i].set_title("Before")
  axes[0,i].axis("off")

  #After pre-processing
  axes[1,i].imshow(data_rgb[i])
  axes[1,i].set_title("After")
  axes[1,i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#splitting the data into train and test sets

In [ ]:
X= data_rgb
y= data_labels

In [ ]:
x_train, x_temp, y_train, y_temp= train_test_split(X, y, test_size= 0.4, stratify= y, random_state= 812)
x_val, x_test, y_val, y_test= train_test_split(x_temp, y_temp, test_size= 0.5, stratify= y_temp, random_state= 812)

In [ ]:
x_val.shape, x_test.shape, x_train.shape

In [ ]:
y_val.shape, y_test.shape, y_train.shape

In [ ]:
x_train= preprocess_input(x_train)
x_val= preprocess_input(x_val)

In [ ]:
x_train.shape

In [ ]:
le= LabelEncoder()
y_train= le.fit_transform(y_train)
y_val= le.transform(y_val)

In [ ]:
y_train= np.array(y_train)
y_val= np.array(y_val)

In [ ]:
base_model= VGG16(
    weights= 'imagenet',
    include_top= False,
    input_shape= (128, 128, 3)
)

In [ ]:
base_model.trainable= False

In [ ]:
model= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation= 'relu'),
    layers.Dropout(0.5),

    layers.Dense(128, activation= 'relu'),
    layers.Dropout(0.3),

    layers.Dense(12, activation= 'softmax')

])

In [ ]:
model.compile(
    optimizer= 'adam',
    loss= 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history= model.fit(
    x_train,
    y_train,
    epochs= 10,
    validation_data= [x_val, y_val])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.legend(['train', 'val'])
plt.title("Accuracy")
plt.show()

In [ ]:
y_pred = model.predict(x_val)
y_pred_classes = np.argmax(y_pred, axis=1)

In [ ]:
cm = confusion_matrix(y_val, y_pred_classes)

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (12 Classes)")
plt.show()

### **Class imbalance check**

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(10,5))
plt.bar(unique, counts)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.show()

### **Data Augmentation**

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
datagen.fit(x_train)

In [ ]:
history= model.fit(datagen.flow(x_train, y_train, batch_size=32),
          validation_data=(x_val, y_val),
          epochs=10)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.legend(['train', 'val'])
plt.title("Accuracy")
plt.show()

**Actionable Insights and Business Recommendations**

1. Model Can Enable Early Weed Detection-
The model can classify seedlings at an early growth stage.
- Business Impact
- Faster intervention
- Reduced weed spread
- Lower crop competition
Deploy the model in a mobile app or field camera system to identify weeds within days of emergence.

2. Confused Classes Need More Training Data
- From the confusion matrix, some plant species will likely be misclassified more often than others.

3. Business Impact
- Misclassification may lead to incorrect treatment recommendations.

4. Recommendation
- Identify the most confused classes and:
- Collect additional images
- Capture multiple lighting conditions
- Include different growth stages
This improves recall for rare but economically important weeds.

5. Transfer Learning Reduces Development cost.